# Bronze Layer — CSV Ingestion Best Practices TEST

**Source**: `/Volumes/learning_sql/bronze/20260716_csv_input_01`

| Approach | Schema | Method | Key Feature |
|---|---|---|---|
| SQL | `learning_sql.bronze_sql` | `COPY INTO` | Idempotent, file-level tracking via Delta log |
| Python | `learning_sql.bronze_python` | Auto Loader (`cloudFiles`) | Scalable, schema evolution, exactly-once |

Both approaches retain raw data without transformations and add:
* `_rescued_data` — captures data that doesn't match the declared schema (schema drift)
* `_ingestion_timestamp` — UTC timestamp of when the row was ingested
* `_source_file` — full path of the source file

In [0]:
%sql
-- ============================================================
-- BRONZE SQL — read_files() + CREATE OR REPLACE TABLE
-- The source folder contains 8 CSV files with different schemas
-- (clients, invoices, products, stores, countries).
-- read_files() merges all schemas automatically via schema
-- evolution — COPY INTO subquery form cannot do this because
-- it is scoped to the first file's schema only.
-- CREATE OR REPLACE TABLE is idempotent: safe to re-run.
-- ============================================================

CREATE SCHEMA IF NOT EXISTS learning_sql.bronze_sql
  COMMENT 'Bronze layer — raw CSV ingestion via SQL';

CREATE OR REPLACE TABLE learning_sql.bronze_sql.sales_raw
USING DELTA
COMMENT 'Raw sales events — bronze layer (SQL / read_files)'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
)
AS
SELECT
  *,
  current_timestamp() AS _ingestion_timestamp,
  _metadata.file_path AS _source_file
FROM read_files(
  '/Volumes/learning_sql/bronze/20260716_csv_input_01',
  format             => 'csv',
  header             => true,
  rescuedDataColumn  => '_rescued_data',
  schemaEvolutionMode => 'addNewColumns'
);

SELECT * FROM learning_sql.bronze_sql.sales_raw LIMIT 5;

In [0]:
from pyspark.sql.functions import current_timestamp, col

# ============================================================
# BRONZE PYTHON — Auto Loader (cloudFiles)
# Exactly-once semantics via checkpoint.
# Scales to billions of files; detects new files automatically.
# trigger(availableNow=True) = batch semantics: process all
# pending files then stop (safe for scheduled jobs).
# ============================================================

SOURCE_PATH     = "/Volumes/learning_sql/bronze/20260716_csv_input_01"
TARGET_TABLE    = "learning_sql.bronze_python.sales_raw"
# Only one volume exists: 20260716_csv_input_01. State is stored as subdirectories
# within the source volume. pathGlobFilter = "*.csv" ensures Auto Loader's file
# discovery ignores the state files stored there.
SCHEMA_LOC      = "/Volumes/learning_sql/bronze/20260716_csv_input_01/_autoloader/schema/sales_raw"
CHECKPOINT_PATH = "/Volumes/learning_sql/bronze/20260716_csv_input_01/_autoloader/checkpoints/sales_raw"

spark.sql("""
  CREATE SCHEMA IF NOT EXISTS learning_sql.bronze_python
  COMMENT 'Bronze layer — raw CSV ingestion via Auto Loader (Python)'
""")

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("rescuedDataColumn", "_rescued_data")           # Captures schema drift
    .option("cloudFiles.inferColumnTypes", "true")          # Infer proper types, not all-string
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # Evolve on new source columns
    .option("cloudFiles.schemaLocation", SCHEMA_LOC)        # Persists inferred schema across runs
    .option("pathGlobFilter", "*.csv")                       # Exclude state files from file discovery
    .load(SOURCE_PATH)
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))  # SC-safe: no input_file_name()
)

(
    df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
    .awaitTermination()
)

display(spark.table(TARGET_TABLE).limit(5))

## Silver Layer — Enriched Sales

**Schema**: `learning_sql.silver` | **Table**: `sales_enriched`

Joins all 8 source files into a unified, enriched sales table (one row per line item):

| Channel | Join path |
|---|---|
| POS | `invoices_pos` → `clients_pos`, `stores`, `countries`, `products_pos` |
| Online | `invoices_online` → `clients_online`, `countries`, `products_online` → `products_pos` |

* `channel` column identifies POS vs Online rows
* `amount` for POS is derived: `unit_price × quantity × (1 − discount_prc / 100)`
* Online products resolve via `products_online` bridge (product_online_id → product_id)
* Country comes from the **store** for POS, from the **client** for Online

In [0]:
%sql
-- ============================================================
-- SILVER - Unified Enriched Sales
-- Reads source CSVs directly and joins all 8 files:
--   POS:    invoices_pos -> clients_pos, stores, countries, products_pos
--   Online: invoices_online -> clients_online, countries,
--           products_online (bridge) -> products_pos
-- Output: one row per line-item, all dimensions resolved.
-- CREATE OR REPLACE TABLE is idempotent: safe to re-run.
-- ============================================================

CREATE SCHEMA IF NOT EXISTS learning_sql.silver
  COMMENT 'Silver layer - cleansed and enriched data';

CREATE OR REPLACE TABLE learning_sql.silver.sales_enriched
USING DELTA
COMMENT 'Unified sales line items - POS and Online, all dimensions joined'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
)
AS
WITH
  -- Source files (read from volume directly)
  invoices_pos    AS (SELECT * FROM read_files('/Volumes/learning_sql/bronze/20260716_csv_input_01/invoices_pos.csv',    format => 'csv', header => true)),
  invoices_online AS (SELECT * FROM read_files('/Volumes/learning_sql/bronze/20260716_csv_input_01/invoices_online.csv', format => 'csv', header => true)),
  clients_pos     AS (SELECT * FROM read_files('/Volumes/learning_sql/bronze/20260716_csv_input_01/clients_pos.csv',     format => 'csv', header => true)),
  clients_online  AS (SELECT * FROM read_files('/Volumes/learning_sql/bronze/20260716_csv_input_01/clients_online.csv',  format => 'csv', header => true)),
  products_pos    AS (SELECT * FROM read_files('/Volumes/learning_sql/bronze/20260716_csv_input_01/products_pos.csv',    format => 'csv', header => true)),
  products_online AS (SELECT * FROM read_files('/Volumes/learning_sql/bronze/20260716_csv_input_01/products_online.csv', format => 'csv', header => true)),
  stores          AS (SELECT * FROM read_files('/Volumes/learning_sql/bronze/20260716_csv_input_01/stores.csv',          format => 'csv', header => true)),
  countries       AS (SELECT * FROM read_files('/Volumes/learning_sql/bronze/20260716_csv_input_01/countries.csv',       format => 'csv', header => true)),

  -- POS: country from store; amount derived from unit_price, quantity, discount
  pos AS (
    SELECT
      'POS'                                                               AS channel,
      ip.date,
      ip.client_id,
      cp.client_name,
      s.store_id,
      s.store_name,
      s.store_country_code                                                AS country_code,
      c.country_name,
      s.store_region,
      ip.product_id,
      pp.product_name,
      pp.product_category,
      ip.quantity,
      ip.unit_price,
      ip.discount_prc,
      ROUND(ip.unit_price * ip.quantity * (1 - ip.discount_prc / 100.0), 2) AS amount
    FROM      invoices_pos ip
    LEFT JOIN clients_pos   cp ON ip.client_id           = cp.client_id
    LEFT JOIN stores         s ON ip.store_id             = s.store_id
    LEFT JOIN countries      c ON s.store_country_code   = c.country_code
    LEFT JOIN products_pos  pp ON ip.product_id           = pp.product_id
  ),

  -- Online: country from client; product resolved via products_online bridge
  online AS (
    SELECT
      'Online'                                                            AS channel,
      io.date,
      io.client_id,
      co.client_name,
      CAST(NULL AS INT)                                                   AS store_id,
      CAST(NULL AS STRING)                                                AS store_name,
      co.client_country_code                                              AS country_code,
      c.country_name,
      CAST(NULL AS STRING)                                                AS store_region,
      po.product_id,
      pp.product_name,
      pp.product_category,
      io.quantity,
      CAST(NULL AS DOUBLE)                                                AS unit_price,
      CAST(NULL AS INT)                                                   AS discount_prc,
      io.amount
    FROM      invoices_online io
    LEFT JOIN clients_online  co ON io.client_id          = co.client_id
    LEFT JOIN countries        c ON co.client_country_code = c.country_code
    LEFT JOIN products_online po ON io.product_online_id  = po.product_online_id
    LEFT JOIN products_pos    pp ON po.product_id          = pp.product_id
  )

SELECT *, current_timestamp() AS _ingestion_timestamp FROM pos
UNION ALL
SELECT *, current_timestamp() AS _ingestion_timestamp FROM online;

-- Verify: row counts and revenue by channel
SELECT
  channel,
  COUNT(*)                       AS rows,
  ROUND(SUM(amount), 2)          AS total_revenue,
  COUNT(DISTINCT client_id)      AS unique_clients,
  COUNT(DISTINCT product_id)     AS unique_products
FROM learning_sql.silver.sales_enriched
GROUP BY channel
ORDER BY channel;